<a href="https://colab.research.google.com/github/Isaque-BD/FATEC-API-6-SEMESTRE/blob/spike-us-20-calculus-pt-and-pnt-by-group/_PT_e_PNT_por_Conjunto_Sem_Gr%C3%A1fico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Perda Análise e Integração das Planilhas SSDMT, CONJ e CTMT

Este notebook realiza o pipeline completo de tratamento e linkagem das três planilhas.

In [ ]:
import pandas as pd
import os

!pip install openpyxl

# Updated to read files from the Colab environment
ssdmt = pd.read_excel('/content/SSDMT.xlsx')
conj = pd.read_excel('/content/CONJ.xlsx')
ctmt = pd.read_excel('/content/CTMT.xlsx')

## Etapa 1 — Tratamento da planilha SSDMT

Selecionar as colunas `CTMT` e `CONJ`, mantendo apenas valores distintos.

In [ ]:
ssdmt_tratado = (
    ssdmt[['CTMT', 'CONJ']].drop_duplicates().reset_index(drop=True)
)

print(f'Registros distintos SSDMT: {len(ssdmt_tratado)}')
ssdmt_tratado.head(10)

Registros distintos SSDMT: 706


,CTMT,CONJ
0,ADT01L3,15845
1,MCA01L1,15857
2,BRT01C3,13251
3,MRG01C4,16567
4,AQZ01I1,15853
5,ESB01S4,15858
6,ART01N5,13242
7,BBL01M5,13248
8,MTI01P4,13332
9,PCJ01P5,13355


## Etapa 2 — Linkagem com a planilha CONJ

Linkar a coluna `COD_ID` da planilha CONJ com a coluna `CONJ` do SSDMT tratado,
retornando a coluna `NOM` e unindo as tabelas.

In [ ]:
conj_select = conj[['COD_ID', 'NOM']].rename(columns={'COD_ID': 'CONJ'})

ssdmt_conj = ssdmt_tratado.merge(conj_select, on='CONJ', how='left')

print(f'Registros após join SSDMT + CONJ: {len(ssdmt_conj)}')
ssdmt_conj.head(10)

Registros após join SSDMT + CONJ: 706


,CTMT,CONJ,NOM
0,ADT01L3,15845,ALDEOTA
1,MCA01L1,15857,MACAOCA
2,BRT01C3,13251,BATURIT?
3,MRG01C4,16567,MARANGUAPE
4,AQZ01I1,15853,AQUIRAZ
5,ESB01S4,15858,EUS?BIO
6,ART01N5,13242,ARACATI
7,BBL01M5,13248,BARBALHA
8,MTI01P4,13332,MAURITI
9,PCJ01P5,13355,PACAJUS


## Etapa 3 — Tratamento da planilha CTMT

### 3a) Somar colunas de Perda Total

As colunas abaixo são somadas em uma única coluna chamada **Perda Total**:

`PERD_A3a, PERD_A4, PERD_B, PERD_MED, PERD_A3a_B, PERD_A4_B, PERD_B_A3a, PERD_B_A4,
PNTMT_01..12, PNTBT_01..12, DESCR, PERD_A3aA4, PERD_A4A3a`

In [ ]:
colunas_PT = [
    'PERD_A3a',
    'PERD_A4',
    'PERD_B',
    'PERD_MED',
    'PERD_A3a_B',
    'PERD_A4_B',
    'PERD_B_A3a',
    'PERD_B_A4',
    'PERD_A3aA4',
    'PERD_A4A3a',
]

colunas_PNT = [
    'PNTMT_01',
    'PNTMT_02',
    'PNTMT_03',
    'PNTMT_04',
    'PNTMT_05',
    'PNTMT_06',
    'PNTMT_07',
    'PNTMT_08',
    'PNTMT_09',
    'PNTMT_10',
    'PNTMT_11',
    'PNTMT_12',
    'PNTBT_01',
    'PNTBT_02',
    'PNTBT_03',
    'PNTBT_04',
    'PNTBT_05',
    'PNTBT_06',
    'PNTBT_07',
    'PNTBT_08',
    'PNTBT_09',
    'PNTBT_10',
    'PNTBT_11',
    'PNTBT_12',
]

# Filtrar apenas as colunas numéricas (excluir DESCR que é texto)
colunas_PT_num = [
    c
    for c in colunas_PT
    if c in ctmt.columns and pd.api.types.is_numeric_dtype(ctmt[c])
]
colunas_PNT_num = [
    c
    for c in colunas_PNT
    if c in ctmt.columns and pd.api.types.is_numeric_dtype(ctmt[c])
]

ctmt['Perda Técnica'] = ctmt[colunas_PT_num].sum(axis=1)
ctmt['Perda Não Técnica'] = ctmt[colunas_PNT_num].sum(axis=1)

# Converter para MWh
ctmt['Perda Técnica (MWh)'] = ctmt['Perda Técnica'] / 1000
ctmt['Perda Não Técnica (MWh)'] = ctmt['Perda Não Técnica'] / 1000

ctmt[['COD_ID', 'Perda Técnica (MWh)', 'Perda Não Técnica (MWh)']].head(5)

,COD_ID,Perda Técnica (MWh),Perda Não Técnica (MWh)
0,ACA01CA,0.0000,0.00000
1,ACA01C1,6208.5030,555.01361
2,ACA01C2,2937.0180,312.63609
3,ACA01C3,3310.7865,500.58351
4,ACA01C4,3995.6490,378.60621


### 3b) Somar colunas de Energia Injetada

As colunas `ENE_01` a `ENE_12` são somadas em uma única coluna chamada **Energia Injetada**.

In [ ]:
colunas_ene = [
    'ENE_01',
    'ENE_02',
    'ENE_03',
    'ENE_04',
    'ENE_05',
    'ENE_06',
    'ENE_07',
    'ENE_08',
    'ENE_09',
    'ENE_10',
    'ENE_11',
    'ENE_12',
]

ctmt['Energia Injetada'] = ctmt[colunas_ene].sum(axis=1)

# Converter para MWh
ctmt['Energia Injetada (Mwh)'] = ctmt['Energia Injetada'] / 1000

ctmt[['COD_ID', 'Energia Injetada (Mwh)']].head(5)

,COD_ID,Energia Injetada (Mwh)
0,ACA01CA,1546.52366
1,ACA01C1,37978.98147
2,ACA01C2,23003.72962
3,ACA01C3,39331.00340
4,ACA01C4,25390.65433


## Etapa 4 — Linkagem Final

Linkar a coluna `COD_ID` da planilha CTMT com a coluna `CTMT` da tabela SSDMT+CONJ.

O resultado é a tabela unificada final.

In [ ]:
ctmt_select = ctmt[
    [
        'COD_ID',
        'Perda Não Técnica (MWh)',
        'Perda Técnica (MWh)',
        'Energia Injetada (Mwh)',
    ]
].rename(columns={'COD_ID': 'CTMT'})

tabela_final = ssdmt_conj.merge(ctmt_select, on='CTMT', how='left')

# Agrupamento por Conjunto
tabela_agrupada = (
    tabela_final
    .groupby('NOM')[
        [
            'Perda Não Técnica (MWh)',
            'Perda Técnica (MWh)',
            'Energia Injetada (Mwh)',
        ]
    ]
    .sum()
    .reset_index()
)

# % Perda Técnica
tabela_agrupada['% PT'] = (
    tabela_agrupada['Perda Técnica (MWh)']
    / tabela_agrupada['Energia Injetada (Mwh)'].replace(0, pd.NA)
) * 100

# % Perda Não Técnica
tabela_agrupada['% PNT'] = (
    tabela_agrupada['Perda Não Técnica (MWh)']
    / tabela_agrupada['Energia Injetada (Mwh)'].replace(0, pd.NA)
) * 100

print(f'Registros na tabela final: {len(tabela_agrupada)}')
print(f'Colunas: {tabela_agrupada.columns.tolist()}')
tabela_agrupada.head(10)

Registros na tabela final: 113
Colunas: ['NOM', 'Perda Não Técnica (MWh)', 'Perda Técnica (MWh)', 'Energia Injetada (Mwh)', '% PT', '% PNT']


,NOM,Perda Não Técnica (MWh),Perda Técnica (MWh),Energia Injetada (Mwh),% PT,% PNT
0,ACARA?,1746.83942,16451.9565,127250.89248,12.928755,1.372752
1,ACARAPE,2020.50454,7641.0810,80626.97752,9.477077,2.505991
2,ACOPIARA,628.43902,7148.4105,54550.99637,13.104088,1.152021
3,AGUA FRIA,24356.19712,22511.5275,330822.45056,6.804716,7.362317
4,ALDEOTA,43197.71028,14980.1820,349426.12319,4.287081,12.362473
5,AMONTADA,573.80351,6878.3400,44878.39114,15.326619,1.278574
6,ANTONINA DO NORTE,287.60125,7495.7085,47257.69329,15.861351,0.608581
7,APUIAR?S,19.93108,3402.5880,25866.40695,13.154467,0.077054
8,AQUIRAZ,2163.92710,7717.9200,114841.32300,6.720508,1.884276
9,ARACATI,3024.78354,13644.5295,160166.45995,8.518968,1.888525
